# ICCIT2026 segpriors — Kaggle worker 2

Runs this account's slice of the channel-mode study (`ICCIT2026_MASTER_PLAN.md`) on **ClinicDB + ISIC18**.

**Assigned configs, in priority order (train x3 seeds [1337, 2024, 7] + eval each), with the
estimated hours behind this assignment (real measurements from other devices in this study —
ClinicDB MK-UNet-T ~0.95h/config, ClinicDB U-Net ~3.87h/config,
ISIC18 MK-UNet-T ~1.46h/config):**
- `experiment/iccit/mkunet_m1_isic18.yaml` (~3.87h)
- `experiment/iccit/mkunet_m2_isic18.yaml` (~3.87h)
- `experiment/iccit/mkunet_m3_isic18.yaml` (~1.46h)
- `experiment/iccit/mkunet_m4_isic18.yaml` (~1.46h)

**Total estimated: ~9.20h**, against a 9.5h internal budget (see the "Run this notebook's assigned configs" cell below)
under a ~10h session target.

**Before running — Kaggle notebook settings:**
1. Settings -> Accelerator -> **GPU T4 x2** (or P100 — anything with CUDA works).
2. Settings -> Internet -> **On** (needed to clone the repo and install packages).
3. Add Data -> attach your ClinicDB dataset and your ISIC18 dataset (the "Attach the ... dataset" cell(s) below search `/kaggle/input/` for them, so exact dataset name/owner doesn't matter).
4. Run all cells top to bottom.

Everything is pinned to commit `4be1071` of the repo so every device in this study runs identical code.


In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
print(subprocess.run(["python3", "--version"], capture_output=True, text=True).stdout)


name, memory.total [MiB]
Tesla P100-PCIE-16GB, 16384 MiB
Python 3.12.13



## 1. Clone the repo at the pinned commit

In [2]:
%cd /kaggle/working
!rm -rf segpriors
!git clone https://github.com/Syfur007/segpriors.git segpriors
%cd segpriors


/kaggle/working
Cloning into 'segpriors'...
remote: Enumerating objects: 375, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 375 (delta 162), reused 351 (delta 138), pack-reused 0 (from 0)
Receiving objects: 100% (375/375), 329.57 KiB | 10.63 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/kaggle/working/segpriors


## 2. Reproduce the exact training environment (Python 3.8 + pinned deps)

Kaggle's default image is a newer Python than this repo's pinned stack (`requirements.txt` is
frozen against Python 3.8 — `pyarrow==17.0.0` is explicitly the last release with a 3.8 wheel).
Rather than fight version resolution against Kaggle's default interpreter, this builds a matching
Miniconda env once, so every device in the study (this notebook, the others, and the SSH remote)
runs the same interpreter + package versions — not just "close enough".

In [3]:
import os
if not os.path.exists("/opt/conda_iccit"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-py38_23.11.0-2-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p /opt/conda_iccit
!/opt/conda_iccit/bin/conda --version


PREFIX=/opt/conda_iccit
Unpacking payload ...
                                                                                
Installing base environment...





Preparing transaction: done
Executing transaction: done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/conda_iccit
conda 23.11.0


In [4]:
PY = "/opt/conda_iccit/bin/python"
PIP = "/opt/conda_iccit/bin/pip"

# --prefer-binary: a few requirements.txt packages (simpleitk especially —
# it's a huge C++ library wrapping ITK, unpinned here — also pyarrow/
# opencv-python/h5py/onnx) fall back to compiling from source if pip can't
# find a prebuilt wheel for the resolved version, which can silently burn
# 20-40+ minutes of this notebook's time budget.
#
# Deliberately NOT run with -q: a prior version of this notebook silently
# ended up missing loguru and tensorboard after this exact install step —
# with no error surfaced, because -q suppresses the resolver output that
# would have shown it. Full output costs nothing here that a broken
# environment discovered hours later wouldn't cost far more of.
!{PIP} install --upgrade pip
!{PIP} install --prefer-binary torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!{PIP} install --prefer-binary -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 23.3.1
    Uninstalling pip-23.3.1:
      Successfully uninstalled pip-23.3.1
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu113
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 16.4 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 127.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of medpy to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing

In [5]:
!pip install loguru tensorboard

  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
Using cached loguru-0.7.3-py3-none-any.whl (61 kB)


### 2b. Verify every critical import actually landed

Checked explicitly rather than assumed — this is exactly the class of failure (a quietly-missing
package) that step 2 above already caused once.

In [6]:
CRITICAL_IMPORTS = [
    "torch", "torchvision", "numpy", "scipy", "sklearn", "skimage", "pandas",
    "loguru", "yaml", "cv2", "albumentations", "pydantic", "pyarrow",
    "statsmodels", "fvcore", "SimpleITK", "nibabel", "h5py", "tensorboard",
    "tensorboardX", "timm", "transformers", "captum", "thop",
]
failed = []
for mod in CRITICAL_IMPORTS:
    result = os.system(f'{PY} -c "import {mod}" 2>/tmp/import_err.txt')
    if result != 0:
        err = open("/tmp/import_err.txt").read().strip().splitlines()[-1:]
        failed.append((mod, err[0] if err else "unknown error"))
        print(f"  MISSING: {mod:20s} {err[0] if err else ''}")
    else:
        print(f"  ok:      {mod}")

if failed:
    raise RuntimeError(
        f"{len(failed)} package(s) failed to import after installation: {[m for m, _ in failed]}. "
        "Re-run the pip install cell above and check its full (non-quiet) output for why."
    )
print("\nAll critical imports OK.")

import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())


  ok:      torch
  ok:      torchvision
  ok:      numpy
  ok:      scipy
  ok:      sklearn
  ok:      skimage
  ok:      pandas
  ok:      loguru
  ok:      yaml
  ok:      cv2
  ok:      albumentations
  ok:      pydantic
  ok:      pyarrow
  ok:      statsmodels
  ok:      fvcore
  ok:      SimpleITK
  ok:      nibabel
  ok:      h5py
  ok:      tensorboard
  ok:      tensorboardX
  ok:      timm
  ok:      transformers
  ok:      captum
  ok:      thop

All critical imports OK.
torch 2.10.0+cu128 cuda available: True


## 3. Attach the ClinicDB dataset

Kaggle mounts an attached dataset as an already-extracted, **read-only** directory tree under
`/kaggle/input/` — not a zip to unpack (e.g.
`/kaggle/input/datasets/<owner>/clinicdb-train-val-test-images-and-masks/ClinicDB/`). This searches
for a directory named `ClinicDB` containing `train/images` so the exact owner/slug doesn't matter,
then symlinks it into place (no copy needed — nothing in this pipeline writes back into the dataset
root).

In [7]:
import glob

candidates = [
    p for p in glob.glob("/kaggle/input/**/ClinicDB", recursive=True)
    if os.path.isdir(os.path.join(p, "train", "images"))
]
assert candidates, (
    "No ClinicDB/ directory (containing train/images) found under /kaggle/input/. "
    "Add the dataset via the '+ Add Data' button (top right) before running this cell."
)
clinicdb_path = candidates[0]
print("Using:", clinicdb_path)

os.makedirs("data/polyp", exist_ok=True)
link_path = "data/polyp/ClinicDB"
if os.path.islink(link_path) or os.path.exists(link_path):
    os.remove(link_path) if os.path.islink(link_path) else None
os.symlink(clinicdb_path, link_path)

assert os.path.isdir("data/polyp/ClinicDB/train/images"), "Unexpected layout — check data/polyp/ClinicDB/"
print("OK — data/polyp/ClinicDB is ready.")
!ls data/polyp/ClinicDB


Using: /kaggle/input/datasets/syfur007/clinicdb-train-val-test-images-and-masks/ClinicDB
OK — data/polyp/ClinicDB is ready.
test			train			 val
test_list_clinicdb.txt	train_list_clinicdb.txt  val_list_clinicdb.txt


## 4. Attach the ISIC18 dataset

Same mounting behaviour as ClinicDB — an already-extracted read-only directory under
`/kaggle/input/`. Searches for a directory named `ISIC2018_Task1-2_Training_Input` (train images)
as the anchor, then symlinks the whole parent directory to `data/isic18_256` (the pre-resized
256x256 layout this study's configs expect — see `scripts/resize_isic18.py` / `configs/dataset/isic18.yaml`
for why it's resized, not the ~13GB raw official download).

In [8]:
import glob

candidates = [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ISIC2018_Task1-2_Training_Input", recursive=True)
]
assert candidates, (
    "No ISIC2018_Task1-2_Training_Input/ directory found under /kaggle/input/. "
    "Add the pre-resized ISIC18 dataset via '+ Add Data' before running this cell — "
    "it must already be resized to 256x256 (scripts/resize_isic18.py), not the raw official download."
)
isic18_path = candidates[0]
print("Using:", isic18_path)

link_path = "data/isic18_256"
if os.path.islink(link_path) or os.path.exists(link_path):
    os.remove(link_path) if os.path.islink(link_path) else None
os.symlink(isic18_path, link_path)

assert os.path.isdir("data/isic18_256/ISIC2018_Task1-2_Training_Input"), "Unexpected layout — check data/isic18_256/"
print("OK — data/isic18_256 is ready.")
!ls data/isic18_256


Using: /kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation
OK — data/isic18_256 is ready.
ISIC2018_Task1-2_Test_Input	 ISIC2018_Task1-2_Validation_Input
ISIC2018_Task1-2_Training_Input  ISIC2018_Task1_Training_GroundTruth


## 5. Attach the BUSI dataset

Same mounting behaviour as ClinicDB/ISIC18 — an already-extracted, **read-only** directory tree
under `/kaggle/input/`. Searches for a directory named `Dataset_BUSI_with_GT` containing a
`benign/` subfolder (the dataset's official layout: `benign/`, `malignant/`, `normal/`, each
holding `<case> (<n>).png` images paired with `<case> (<n>)_mask.png` masks), then symlinks it
into place — no resize step needed here, unlike ISIC18: BUSI images average ~500×500 (not
multi-megapixel), so the whole dataset caches at well under 1GB, comfortably inside
`configs/dataset/busi.yaml`'s cache limit.

**Recommended source** — the most-used Kaggle mirror of the original Al-Dhabyani et al. (2020,
*Data in Brief*) release (504 votes / 73.7k downloads / 564 public notebooks as of this writing;
by far the most-used BUSI mirror on Kaggle — the next-closest has 63 votes):
[kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset](https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset)

Attach it via **+ Add Data** before running the cell below. Note `datasets/busi.py`'s mandatory
`dedup()` and `detect_colour_contamination()` preprocessing passes run the first time any BUSI
config actually loads data (inside the sanity check / training cells below, not this attach
cell) — expect a one-time delay there, not here.

In [ ]:
import glob

candidates = [
    p for p in glob.glob("/kaggle/input/**/Dataset_BUSI_with_GT", recursive=True)
    if os.path.isdir(os.path.join(p, "benign"))
]
assert candidates, (
    "No Dataset_BUSI_with_GT/ directory (containing a benign/ subfolder) found under "
    "/kaggle/input/. Add the BUSI dataset via '+ Add Data' before running this cell — "
    "https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset is this "
    "study's recommended source."
)
busi_path = candidates[0]
print("Using:", busi_path)

os.makedirs("data/busi", exist_ok=True)
link_path = "data/busi/Dataset_BUSI_with_GT"
if os.path.islink(link_path) or os.path.exists(link_path):
    os.remove(link_path) if os.path.islink(link_path) else None
os.symlink(busi_path, link_path)

assert os.path.isdir("data/busi/Dataset_BUSI_with_GT/benign"), "Unexpected layout — check data/busi/Dataset_BUSI_with_GT/"
print("OK — data/busi/Dataset_BUSI_with_GT is ready.")
!ls data/busi/Dataset_BUSI_with_GT


## 6. Sanity check (mirrors the pre-flight gate run on the other devices)

In [ ]:
os.environ["PYTHONPATH"] = os.getcwd()
!{PY} -c "
from utils.config import load_config
from datasets import StandardSplitDataModule
cfg = load_config('configs/experiment/iccit/mkunet_m1_clinicdb.yaml')
dm = StandardSplitDataModule(cfg)
tl, vl = dm.get_standard_loaders()
print('train:', len(tl.dataset), 'val:', len(vl.dataset))

## 7. Run this notebook's assigned configs

Each config trains all 3 pre-registered seeds (`[1337, 2024, 7]`) via `orchestration.runner.run_sweep`
(writes `artifacts/runs/<run_id>/manifest.json` + ledger rows, exactly like the other devices), then
evaluates each seed's checkpoint. One config failing does not stop the rest.

Configs are listed **in priority order** (heaviest/most-important first) with a per-config hour
estimate attached. Before starting each one, the remaining 9.5h budget is checked
against that config's estimate x1.4 (real per-seed ISIC18 times varied
0.35-0.79h in measurements from this same study, so gating on the bare average risked a config
getting killed mid-run by Kaggle's session limit instead of skipped cleanly beforehand) — if it
doesn't clearly fit, it's skipped and the *next* (possibly smaller) config is tried instead, so a
big item near the end doesn't block smaller ones after it. Anything skipped just doesn't run —
nothing is left half-written for this study's manifest/ledger contract to choke on.

In [ ]:
import time

CONFIGS = [
    {
        "path": "experiment/iccit/mkunet_m1_clinicdb.yaml",
        "hours": 3.87
    },
    {
        "path": "experiment/iccit/mkunet_m2_clinicdb.yaml",
        "hours": 3.87
    },
    {
        "path": "experiment/iccit/mkunet_m3_clinicdb.yaml",
        "hours": 1.46
    },
    {
        "path": "experiment/iccit/mkunet_m4_clinicdb.yaml",
        "hours": 1.46
    },
    {
        "path": "experiment/iccit/mkunet_m5_clinicdb.yaml",
        "hours": 1.46
    },
    {
        "path": "experiment/iccit/mkunet_m6_clinicdb.yaml",
        "hours": 1.46
    },
    {
        "path": "experiment/iccit/mkunet_m7_clinicdb.yaml",
        "hours": 1.46
    },
    {
        "path": "experiment/iccit/mkunet_m8_clinicdb.yaml",
        "hours": 1.46
    }
]
BUDGET_SECONDS = 9.5 * 3600
SAFETY_MULTIPLIER = 1.4
start_time = time.time()

results = []
for entry in CONFIGS:
    cfg, est_hours = entry["path"], entry["hours"]
    elapsed = time.time() - start_time
    remaining_hours = (BUDGET_SECONDS - elapsed) / 3600
    print(f"\n[{elapsed/3600:.2f}h elapsed, {remaining_hours:.2f}h left, next={cfg} (est {est_hours}h)]")

    if remaining_hours < est_hours * SAFETY_MULTIPLIER:
        print(f"  Not enough safety-padded budget for {cfg} (needs ~{est_hours * SAFETY_MULTIPLIER:.2f}h) — skipping.")
        results.append((cfg, "skipped (budget)", "skipped (budget)"))
        continue

    print(f"{'='*70}\n TRAIN {cfg}\n{'='*70}")
    rc_train = os.system(f"{PY} scripts/run_iccit_sweep.py --config configs/{cfg}")
    print(f"{'='*70}\n EVAL {cfg}\n{'='*70}")
    rc_eval = os.system(f"{PY} scripts/eval_iccit_sweep.py --config configs/{cfg}")
    results.append((cfg, rc_train, rc_eval))

print(f"\n\n=== SUMMARY (0 = ok, nonzero = at least one seed failed) — total {(time.time()-start_time)/3600:.2f}h ===")
for cfg, rc_t, rc_e in results:
    print(f"  {cfg:55s} train={rc_t} eval={rc_e}")


## 8. Package results for download

Kaggle keeps `/kaggle/working/` as this notebook version's Output after the session ends —
download `iccit_results_worker2.zip` from the Output tab once this finishes.

In [ ]:
!cd /kaggle/working/segpriors && zip -qr /kaggle/working/iccit_results_worker2.zip artifacts/ checkpoints/ logs/ -x "*.pth"
!cd /kaggle/working/segpriors && zip -qr /kaggle/working/iccit_checkpoints_worker2.zip checkpoints/
!ls -lh /kaggle/working/*.zip
